<a href="https://colab.research.google.com/github/DarkfinShark/compling-hw/blob/main/hw3_w2v.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.2 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

In [22]:
# Выводим название всех датасетов и моделей gensim
api.info(name_only=True)

{'corpora': ['semeval-2016-2017-task3-subtaskBC',
  'semeval-2016-2017-task3-subtaskA-unannotated',
  'patent-2017',
  'quora-duplicate-questions',
  'wiki-english-20171001',
  'text8',
  'fake-news',
  '20-newsgroups',
  '__testing_matrix-synopsis',
  '__testing_multipart-matrix-synopsis'],
 'models': ['fasttext-wiki-news-subwords-300',
  'conceptnet-numberbatch-17-06-300',
  'word2vec-ruscorpora-300',
  'word2vec-google-news-300',
  'glove-wiki-gigaword-50',
  'glove-wiki-gigaword-100',
  'glove-wiki-gigaword-200',
  'glove-wiki-gigaword-300',
  'glove-twitter-25',
  'glove-twitter-50',
  'glove-twitter-100',
  'glove-twitter-200',
  '__testing_word2vec-matrix-synopsis']}

1. 'semeval-2016-2017-task3-subtaskA-unannotated': данные с форума Quatar Living; 189941 вопросов с форума +комментарии к ним; можно использовать для обучения моделей генерации ответов на вопросы.
2. 'fake-news': посты и метаинформация с 244 сайтов, распознанные BS Detector как ненадёжные; 12999 постов; может использоваться для обучения моделей распознавания фейковых новостей.
3. 'patent-2017': полные тексты заявок на патенты за 2017 год; 353197 документов; можно использовать для обучения моделей извлечения и структурирования  информации из текстов патента.

**Базовые операции с векторами**

In [9]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [10]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [23]:
# загрузка датасета
dataset = api.load('text8')

# загрузка модели
model = api.load('glove-twitter-50')

[==================================================] 100.0% 199.5/199.5MB downloaded


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [16]:
from gensim.models.keyedvectors import KeyedVectors

def find_similar(word: str, model: KeyedVectors, n: int = 10) -> list[str]:
    """Находит наиболее близкие по вектору слова к исходному слову, используя векторные представления из gensim.

    Args:
        word: Исходное слово для поиска похожих.
        model: Объект KeyedVectors с предобученными векторными представлениями слов из gensim.
        n: Количество возвращаемых слов (по умолчанию 10).

    Returns:
        Список n наиболее близких по вектору слов к исходному.

    Raises:
        KeyError: Если исходное слово отсутствует в словаре модели.
    """

    similar_words = model.most_similar(word, topn=n)
    return [word for word, _ in similar_words]


In [25]:
find_similar(word='christmas', model=model)

['xmas',
 'holiday',
 'easter',
 'valentines',
 'day',
 'valentine',
 'merry',
 'holidays',
 'friday',
 'summer']

3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [26]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [43]:
model = Word2Vec(cooking_sentences, vector_size=50, window=3, min_count=1, workers=2, sg=1)

In [44]:
print(f"Слова в словаре: {list(w2v_model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [45]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [46]:
# Находим похожие слова с помощью функции из пункта 2
print("Слова, похожие на 'духовка':")
print(*find_similar(word='духовка', model=model.wv, n=5), sep=', ')
print()
print("Слова, похожие на 'овощи':")
print(*find_similar(word='овощи', model=model.wv, n=5), sep=', ')

Слова, похожие на 'духовка':
ингредиенты, десерт, холодильник, питание, пирог

Слова, похожие на 'овощи':
мариновать, хлеб, гриль, фольга, сахар


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [31]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [47]:
words = ['варить', 'духовка', 'овощи']
for word in words:
    try:
        similar = ft_model.wv.most_similar(word, topn=5)
        print(f"Слова, похожие на {word}:")
        for word, score in similar:
            print(f"  {word}: {score:.4f}")
    except KeyError:
        print(f"Слово {word} не найдено в словаре")

Слова, похожие на варить:
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
Слова, похожие на духовка:
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944
Слова, похожие на овощи:
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [48]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')

# Сравниваем для слов с опечатками
compare_models('броколи')
compare_models('соуч')
compare_models('ингредеенты')


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']

Сравнение для слова: 'броколи'
  Word2Vec: слово не найдено
  FastText: ['брокколи', 'молоко']

Сравнение для слова: 'соуч'
  Word2Vec: слово не найдено
  FastText: ['огурцы', 'помидоры']

Сравнение для слова: 'ингредеенты'
  Word2Vec: слово не найдено
  FastText: ['ингредиенты', 'овощи']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [49]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [50]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [51]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [52]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [53]:
compare_documents(2,4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [54]:
doc_model.dv.most_similar("doc_1", topn=1)

[('doc_0', 0.2735169529914856)]

10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [65]:
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

models = [ft_model_10, ft_model_50, ft_model_100]
words = ['индейка', 'паприка', 'вафельница']

for model in models:
    print(model)
    for word in words:
        try:
            similar = model.wv.most_similar(word, topn=5)
            print(f"Слова, похожие на {word}:")
            for word, score in similar:
                print(f"  {word}: {score:.4f}")
        except KeyError:
            print(f"Слово {word} не найдено в словаре")
        print()

FastText<vocab=65, vector_size=10, alpha=0.025>
Слова, похожие на индейка:
  курица: 0.5626
  яичница: 0.5516
  варить: 0.5339
  завтрак: 0.5200
  говядина: 0.5068

Слова, похожие на паприка:
  чай: 0.7258
  мясо: 0.7109
  картофель: 0.5609
  вино: 0.5215
  сахар: 0.5109

Слова, похожие на вафельница:
  вода: 0.5107
  кипятить: 0.5072
  тесто: 0.5049
  масло: 0.5011
  чай: 0.4507

FastText<vocab=65, vector_size=50, alpha=0.025>
Слова, похожие на индейка:
  кофе: 0.2794
  курица: 0.2585
  гриль: 0.2066
  барбекю: 0.2013
  яйца: 0.1997

Слова, похожие на паприка:
  хлеб: 0.3238
  сковорода: 0.2745
  молоко: 0.2428
  гриль: 0.2038
  взбивать: 0.1941

Слова, похожие на вафельница:
  тесто: 0.5023
  яичница: 0.4792
  чашка: 0.2494
  говядина: 0.2360
  дрожжи: 0.2037

FastText<vocab=65, vector_size=100, alpha=0.025>
Слова, похожие на индейка:
  ингредиенты: 0.2554
  десерт: 0.2069
  пирог: 0.1914
  соль: 0.1810
  тушить: 0.1624

Слова, похожие на паприка:
  десерт: 0.2434
  помидоры: 0.2062
